In [ ]:

!pip install -q langchain langchain-groq

import os
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
if not api_key:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets first.")

os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Setup successful!")


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are a friendly AI assistant for AgriFresh Agricultural Products.

FACTS:
Sample demonstration prices per kg:
Rice Rs 60.
Wheat Rs 35.
Maize Rs 30.
Onion Rs 40.
Tomato Rs 30.
These are sample prices for this project, not live market rates.

RULES:
Never invent agricultural products or prices.
Clearly explain that prices are sample rates and may vary by market, quality, season, and location.
If a product or current market rate is not listed, say "I'll check the latest rate with the seller."
Calculate totals carefully.
Never promise a delivery time or place.
Give basic product information only; do not make unsupported claims about quality or availability.

FORMAT:
Under 60 words.
Friendly English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })
    history.extend([
        ("human", question),
        ("ai", answer)
    ])
    return answer

print(ask("Hi, what agricultural products do you sell?"))
print(ask("What is the sample price of rice?"))

In [ ]:
import csv

# Create a small test dataset for the agricultural-products assistant.
test_cases = [
    {"question": "What is the sample price of rice?", "must_include": "60"},
    {"question": "What is the sample price of wheat?", "must_include": "35"},
    {"question": "What is the sample price of maize?", "must_include": "30"},
    {"question": "What is the sample price of onion?", "must_include": "40"},
    {"question": "Are these live market prices?", "must_include": "sample"},
]

with open("tests.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(test_cases)

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += int(ok)

        print(
            "PASS" if ok else "FAIL",
            "|", row["question"],
            "→", answer[:100]
        )

    rate = passed / len(rows) if rows else 0
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")

run_tests()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What agricultural products do you sell?"}
    ]
})

print(result["messages"][-1].content)

In [ ]:
# Agricultural Products AI Assistant using LangChain + Groq.

This project demonstrates an AI assistant that answers customer questions about common agricultural products, sample prices, and basic product information.

In [ ]:
SYSTEM = """ROLE:
You are a friendly AI assistant for AgriFresh, an agricultural products store.

FACTS:
Sample demonstration prices per kg:
Rice Rs 60.
Wheat Rs 35.
Maize Rs 30.
Onion Rs 40.
Tomato Rs 30.
These are sample prices, not live market rates.

RULES:
Never invent product names or prices.
Clearly state that prices are samples and may vary by market, quality, season, and location.
If information or a current market rate is unavailable, say "I'll check the latest rate with the seller."
Calculate totals carefully.
Never promise a delivery time or place.

FORMAT:
Under 60 words.
Friendly English or Telugu matching the customer.
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()
history = []

In [ ]:
print(ask("What is the sample price of rice?"))
print(ask("What is the sample price of wheat?"))
print(ask("Do you have maize?"))

In [ ]:
import os

print("Current folder:", os.getcwd())
print("CSV file exists:", os.path.exists("tests.csv"))

print("\\nAgricultural-products test data:\\n")
with open("tests.csv", "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        if ok:
            passed += 1
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:100])
    total = len(rows)
    print(f"\\n{passed}/{total} passed ({passed/total:.0%})")

run_tests()

In [ ]:
!pip install -q langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=1.0
)
print("Temperature changed to 1.0!")

In [ ]:
chain = prompt | llm | StrOutputParser()

In [ ]:
print("First Run:")
run_tests()

In [ ]:
print("Second Run:")
run_tests()

In [ ]:
jobs = [
    {
        "job": "Calculate the total price of 3 kg of rice",
        "type": "Plain Code"
    },
    {
        "job": "Answer customer questions about agricultural products and sample prices",
        "type": "One LLM Call"
    },
    {
        "job": "Check agricultural product availability or current rates using a tool",
        "type": "Agent"
    }
]
for item in jobs:
    print("Job:", item["job"])
    print("Type:", item["type"])
    print()